# 🏦 Financial Facts Data Processing Pipeline for Power BI

Goal of this notebook:
1. Read raw facts files for each company.
2. Filter important accounting concepts like Revenue, Gross Profit, Assets... etc.
3. Create a Pivot table so each concept becomes its own column.
4. Merge alternatives (e.g., Revenues and RevenueFromContractWithCustomerExcludingAssessedTax) into a single unified column.
5. Calculate Gross Profit intelligently with a fallback plan if data is missing.
6. Save the final clean file ready for Power BI use.

## 1️⃣ Import Libraries

In [1]:
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 150)

## 2️⃣ Load Raw Data File

In [2]:
RAW_DATA_PATH = "K:/Tech Giants Financial Analytics/Data/all_companies_raw_facts.csv"

df = pd.read_csv(RAW_DATA_PATH)

print(f"File loaded successfully ✅  |  Number of rows: {len(df):,}  |  Number of columns: {df.shape[1]}")

File loaded successfully ✅  |  Number of rows: 13,237  |  Number of columns: 10


## 3️⃣ Quick Data Overview

> Note: In the original code there was a line `df.read()` which doesn't exist in pandas
> (It should be `df.head()`), so it was corrected here.

In [3]:
# First 5 rows of data
df.head()

,Company,CIK,Concept,Year,Quarter,Form,Value,Unit,Description,Frame
0,Apple,320193,AccountsPayableCurrent,2021,FY,10-K,4.229600e+10,USD,Carrying value as of the balance sheet date of...,CY2020Q3I
1,Apple,320193,AccountsPayableCurrent,2021,FY,10-K,5.476300e+10,USD,Carrying value as of the balance sheet date of...,NaN
2,Apple,320193,AccountsPayableCurrent,2022,FY,10-K,5.476300e+10,USD,Carrying value as of the balance sheet date of...,CY2021Q3I
3,Apple,320193,AccountsPayableCurrent,2022,FY,10-K,6.411500e+10,USD,Carrying value as of the balance sheet date of...,NaN
4,Apple,320193,AccountsPayableCurrent,2023,FY,10-K,6.411500e+10,USD,Carrying value as of the balance sheet date of...,CY2022Q3I


In [4]:
# General information about columns and data types
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13237 entries, 0 to 13236
Data columns (total 10 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Company      13237 non-null  object 
 1   CIK          13237 non-null  int64  
 2   Concept      13237 non-null  object 
 3   Year         13237 non-null  int64  
 4   Quarter      13237 non-null  object 
 5   Form         13237 non-null  object 
 6   Value        13237 non-null  float64
 7   Unit         13237 non-null  object 
 8   Description  13107 non-null  object 
 9   Frame        6993 non-null   object 
dtypes: float64(1), int64(2), object(7)
memory usage: 1.0+ MB


In [5]:
# Quick descriptive statistics for numeric columns
df.describe()

,CIK,Year,Value
count,1.323700e+04,13237.000000,1.323700e+04
mean,1.049613e+06,2023.047896,2.304710e+10
std,4.460690e+05,1.423947,6.045890e+10
min,3.201930e+05,2021.000000,-1.425450e+11
25%,7.890190e+05,2022.000000,2.180000e+08
50%,1.018724e+06,2023.000000,2.865000e+09
75%,1.326801e+06,2024.000000,1.691200e+10
max,1.652044e+06,2025.000000,8.180420e+11


## 4️⃣ Define Target Accounting Concepts

In [6]:
# List of accounting concepts needed from the XBRL Facts file
target_concepts = [
    "GrossProfit",
    "NetIncomeLoss",
    "Assets",
    "AssetsCurrent",
    "LiabilitiesCurrent",
    "OperatingIncomeLoss",
    "Revenues",
    "RevenueFromContractWithCustomerExcludingAssessedTax",
    "CostOfGoodsAndServicesSold",
    "CostOfRevenue",
]

print("Filtering data for the following concepts (including alternatives):")
for c in target_concepts:
    print(f"  - {c}")

Filtering data for the following concepts (including alternatives):
  - GrossProfit
  - NetIncomeLoss
  - Assets
  - AssetsCurrent
  - LiabilitiesCurrent
  - OperatingIncomeLoss
  - Revenues
  - RevenueFromContractWithCustomerExcludingAssessedTax
  - CostOfGoodsAndServicesSold
  - CostOfRevenue


## 5️⃣ Filter Data

In [7]:
df_filtered = df[df["Concept"].isin(target_concepts)].copy()

print(f"Number of rows after filtering: {len(df_filtered):,}")
df_filtered.head()

Number of rows after filtering: 485


,Company,CIK,Concept,Year,Quarter,Form,Value,Unit,Description,Frame
70,Apple,320193,Assets,2021,FY,10-K,3.238880e+11,USD,Sum of the carrying amounts as of the balance ...,CY2020Q3I
71,Apple,320193,Assets,2021,FY,10-K,3.510020e+11,USD,Sum of the carrying amounts as of the balance ...,NaN
72,Apple,320193,Assets,2022,FY,10-K,3.510020e+11,USD,Sum of the carrying amounts as of the balance ...,CY2021Q3I
73,Apple,320193,Assets,2022,FY,10-K,3.527550e+11,USD,Sum of the carrying amounts as of the balance ...,NaN
74,Apple,320193,Assets,2023,FY,10-K,3.527550e+11,USD,Sum of the carrying amounts as of the balance ...,CY2022Q3I


## 6️⃣ Transform Data to Pivot Table

Each row represents a company + year, and each column represents an accounting concept.
If a concept repeats for the same company in the same year, we take the last value (`aggfunc="last"`).

In [8]:
df_pivoted = df_filtered.pivot_table(
    index=["Company", "CIK", "Year"],
    columns="Concept",
    values="Value",
    aggfunc="last",
).reset_index()

# Remove the pivot column name (Concept) so column names appear as plain text
df_pivoted.columns.name = None

df_pivoted.head()

,Company,CIK,Year,Assets,AssetsCurrent,CostOfGoodsAndServicesSold,CostOfRevenue,GrossProfit,LiabilitiesCurrent,NetIncomeLoss,OperatingIncomeLoss,RevenueFromContractWithCustomerExcludingAssessedTax,Revenues
0,Amazon,1018724,2021,4.205490e+11,1.615800e+11,2.723440e+11,NaN,NaN,1.422660e+11,3.336400e+10,2.487900e+10,4.698220e+11,NaN
1,Amazon,1018724,2022,4.626750e+11,1.467910e+11,2.888310e+11,NaN,NaN,1.553930e+11,-2.722000e+09,1.224800e+10,5.139830e+11,NaN
2,Amazon,1018724,2023,5.278540e+11,1.723510e+11,3.047390e+11,NaN,NaN,1.649170e+11,3.042500e+10,3.685200e+10,5.747850e+11,NaN
3,Amazon,1018724,2024,6.248940e+11,1.908670e+11,3.262880e+11,NaN,NaN,1.794310e+11,5.924800e+10,6.859300e+10,6.379590e+11,NaN
4,Amazon,1018724,2025,8.180420e+11,2.290830e+11,3.564140e+11,NaN,NaN,2.180050e+11,7.767000e+10,7.997500e+10,7.169240e+11,NaN


## 7️⃣ Ensure All Required Columns Exist (Avoid KeyError)

In [9]:
# If a specific concept is completely missing from the data, add it with default value 0.0
for col in target_concepts:
    if col not in df_pivoted.columns:
        df_pivoted[col] = 0.0

# Replace any missing values (NaN) with zero before calculations
df_pivoted.fillna(0, inplace=True)

## 8️⃣ Merge Revenue and Cost Alternatives

Some companies use `Revenues` and some use `RevenueFromContractWithCustomerExcludingAssessedTax`,
so we merge them into a single unified column. Same concept applies to cost of goods sold.

In [10]:
# Merge revenue alternatives
df_pivoted["Total_Revenues"] = (
    df_pivoted["Revenues"]
    + df_pivoted["RevenueFromContractWithCustomerExcludingAssessedTax"]
)

# Merge cost alternatives
df_pivoted["Total_Cost"] = (
    df_pivoted["CostOfGoodsAndServicesSold"] + df_pivoted["CostOfRevenue"]
)

## 9️⃣ Calculate Gross Profit Intelligently

Logical calculation order:
1. If original `GrossProfit` value exists and is not zero → use it as is.
2. If missing, calculate manually: `Revenue - Cost`.
3. If data is completely missing (e.g., Amazon which doesn't disclose Gross Profit),
   estimate it using an assumed profit margin of 43% from revenue (approximate estimate,
   not 100% accurate, but better than leaving it at zero).

In [11]:
ESTIMATED_MARGIN_FALLBACK = 0.43  # Estimated margin used only when all other data is missing


def calculate_gross_profit(row: pd.Series) -> float:
    """Calculate gross profit for a single row (company + year) using a tiered fallback strategy."""

    # Step 1: Use original value if it exists
    if row["GrossProfit"] != 0:
        return row["GrossProfit"]

    # Step 2: Calculate difference between revenue and cost
    calculated_gp = row["Total_Revenues"] - row["Total_Cost"]
    if calculated_gp != 0 and calculated_gp != row["Total_Revenues"]:
        return calculated_gp

    # Step 3: Estimate using assumed profit margin (e.g., Amazon case)
    if row["Total_Revenues"] != 0:
        return row["Total_Revenues"] * ESTIMATED_MARGIN_FALLBACK

    # If no data is available at all
    return 0.0


df_pivoted["Gross_Profit_Final"] = df_pivoted.apply(calculate_gross_profit, axis=1)

## 🔟 Prepare Final Columns for Power BI

In [12]:
# Name columns with clear and meaningful names for any user of the Power BI report
final_columns = {
    "Assets": "Total_Assets",
    "AssetsCurrent": "Current_Assets",
    "LiabilitiesCurrent": "Current_Liabilities",
    "NetIncomeLoss": "Net_Income",
    "OperatingIncomeLoss": "Operating_Income",
    "Total_Revenues": "Total_Revenue",
    "Gross_Profit_Final": "Gross_Profit",
}

df_out = df_pivoted[["Company", "CIK", "Year"] + list(final_columns.keys())].rename(
    columns=final_columns
)

df_out.fillna(0, inplace=True)
df_out.head()

,Company,CIK,Year,Total_Assets,Current_Assets,Current_Liabilities,Net_Income,Operating_Income,Total_Revenue,Gross_Profit
0,Amazon,1018724,2021,4.205490e+11,1.615800e+11,1.422660e+11,3.336400e+10,2.487900e+10,4.698220e+11,1.974780e+11
1,Amazon,1018724,2022,4.626750e+11,1.467910e+11,1.553930e+11,-2.722000e+09,1.224800e+10,5.139830e+11,2.251520e+11
2,Amazon,1018724,2023,5.278540e+11,1.723510e+11,1.649170e+11,3.042500e+10,3.685200e+10,5.747850e+11,2.700460e+11
3,Amazon,1018724,2024,6.248940e+11,1.908670e+11,1.794310e+11,5.924800e+10,6.859300e+10,6.379590e+11,3.116710e+11
4,Amazon,1018724,2025,8.180420e+11,2.290830e+11,2.180050e+11,7.767000e+10,7.997500e+10,7.169240e+11,3.605100e+11


## 1️⃣1️⃣ Save Final File

In [13]:
output_file = "K:/Tech Giants Financial Analytics/Data/powerbi_ready_matrix.csv"
df_out.to_csv(output_file, index=False, encoding="utf-8-sig")

print(f"\n[SUCCESS] Processing completed successfully ✅  |  File saved at: {output_file}")
df_out[["Company", "Year", "Total_Revenue", "Gross_Profit"]].head()


[SUCCESS] Processing completed successfully ✅  |  File saved at: K:/Tech Giants Financial Analytics/Data/powerbi_ready_matrix.csv


,Company,Year,Total_Revenue,Gross_Profit
0,Amazon,2021,4.698220e+11,1.974780e+11
1,Amazon,2022,5.139830e+11,2.251520e+11
2,Amazon,2023,5.747850e+11,2.700460e+11
3,Amazon,2024,6.379590e+11,3.116710e+11
4,Amazon,2025,7.169240e+11,3.605100e+11
